# Решения: KMeans и DBSCAN

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")

from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler

FEATURES = ["delivery_days", "freight_value", "delay_days"]


## Урок. 1–3. Признаки и масштаб

In [ ]:
X = df[FEATURES].copy()
feature_range = X.max() - X.min()
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
assert Xs.shape == X.shape and np.allclose(Xs.mean(axis=0), 0, atol=1e-7)


## Урок. 4–5. KMeans

In [ ]:
kmeans = KMeans(n_clusters=min(3, len(df)), random_state=53, n_init=10)
labels_km = kmeans.fit_predict(Xs)
cluster_sizes = pd.Series(labels_km).value_counts().sort_index()
assert int(cluster_sizes.sum()) == len(df)


## Урок. 6. Описательные профили

In [ ]:
clustered = df.assign(cluster_km=labels_km)
profile_km = clustered.groupby("cluster_km")[FEATURES + ["is_late"]].mean().round(2)
assert len(profile_km) == len(set(labels_km))
print(profile_km)


## Урок. 7–8. DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.9, min_samples=4)
labels_db = dbscan.fit_predict(Xs)
n_noise = int(np.sum(labels_db == -1))
n_db_clusters = len(set(labels_db) - {-1})
assert 0 <= n_noise <= len(df)


## Урок. 9. Эксперимент eps

In [ ]:
eps_rows = []
for eps in (0.5, 0.9, 1.3):
    labels = DBSCAN(eps=eps, min_samples=4).fit_predict(Xs)
    eps_rows.append({"eps": eps, "clusters": len(set(labels) - {-1}), "noise": int(np.sum(labels == -1))})
eps_table = pd.DataFrame(eps_rows)
assert len(eps_table) == 3


## Урок. 10. Интерпретация

In [ ]:
CLUSTER_NOTE = "KMeans разделил заказы на три группы по совместному масштабу срока, стоимости доставки и задержки. Средняя is_late помогает описать уже найденные группы, но не является входным признаком и не превращает анализ в предсказание. DBSCAN отдельно показывает плотные области и редкие точки."
assert len(CLUSTER_NOTE) >= 180


## ДЗ. A1. Перебор k

In [ ]:
k_rows = []
for k in range(2, 6):
    model = KMeans(n_clusters=min(k, len(df)), random_state=53, n_init=10)
    labels = model.fit_predict(Xs)
    k_rows.append({"k": k, "inertia": float(model.inertia_), "smallest_cluster": int(pd.Series(labels).value_counts().min())})
k_table = pd.DataFrame(k_rows)
assert len(k_table) == 4


## ДЗ. A2. Профиль k=4

In [ ]:
model4 = KMeans(n_clusters=min(4, len(df)), random_state=53, n_init=10)
labels4 = model4.fit_predict(Xs)
profile4 = df.assign(cluster=labels4).groupby("cluster")[FEATURES + ["is_late"]].mean()
assert len(profile4) == min(4, len(df))


## ДЗ. A3. Сетка DBSCAN

In [ ]:
db_rows = []
for eps in np.arange(0.6, 1.41, 0.2):
    labels = DBSCAN(eps=float(eps), min_samples=4).fit_predict(Xs)
    db_rows.append({"eps": round(float(eps), 1), "clusters": len(set(labels) - {-1}), "noise": int(np.sum(labels == -1))})
scan_table = pd.DataFrame(db_rows)
assert len(scan_table) == 5


## ДЗ. Challenge

In [ ]:
inertias = []
for seed in range(5):
    model = KMeans(n_clusters=min(3, len(df)), random_state=seed, n_init=10).fit(Xs)
    inertias.append(float(model.inertia_))
MODEL_NOTE = "StandardScaler нужен, чтобы freight_value не подавлял признаки дней. KMeans требует заранее выбрать число групп и даёт сегмент каждой точке. DBSCAN ищет плотные области, может оставить шум и чувствителен к eps. Алгоритм выбирают по операционному вопросу, а параметры проверяют экспериментом."
assert len(inertias) == 5 and len(MODEL_NOTE) >= 220
